In [1]:
import os
import pandas as pd

print("Files in current directory:", os.listdir("."))
# Let's inspect the dataset if it exists
df = pd.read_csv("Used Car Dataset.csv")
print("Columns:", df.columns.tolist())
print("Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print("\nData Info:")
print(df.info())
print("\nMissing Values:")
print(df.isnull().sum())

Files in current directory: ['car_price.py', 'f.ipynb', 'project_ass.ipynb', 'Used Car Dataset.csv']
Columns: ['Unnamed: 0', 'car_name', 'registration_year', 'insurance_validity', 'fuel_type', 'seats', 'kms_driven', 'ownsership', 'transmission', 'manufacturing_year', 'mileage(kmpl)', 'engine(cc)', 'max_power(bhp)', 'torque(Nm)', 'price(in lakhs)']
Shape: (1553, 15)

First 5 rows:
   Unnamed: 0                                           car_name  \
0           0                    2017 Mercedes-Benz S-Class S400   
1           1  2020 Nissan Magnite Turbo CVT XV Premium Opt BSVI   
2           2                       2018 BMW X1 sDrive 20d xLine   
3           3                           2019 Kia Seltos GTX Plus   
4           4                    2019 Skoda Superb LK 1.8 TSI AT   

  registration_year insurance_validity fuel_type  seats  kms_driven  \
0            Jul-17      Comprehensive    Petrol      5       56000   
1            Jan-21      Comprehensive    Petrol      5       3061

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

df = pd.read_csv("Used Car Dataset.csv")

# Clean unnamed column
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

# Clean missing values
df = df.dropna()

# Check unique values and types
print(df.dtypes)
print(df.head())

# Manufacturing year to numeric if needed
df['manufacturing_year'] = pd.to_numeric(df['manufacturing_year'], errors='coerce')
df = df.dropna()

# Let's see categorical columns
cat_cols = ['insurance_validity', 'fuel_type', 'ownsership', 'transmission']
print("\nCategorical columns unique values:")
for col in cat_cols:
    print(col, df[col].unique())

# Prepare features and target
features = ['registration_year', 'insurance_validity', 'fuel_type', 'seats', 'kms_driven', 
            'ownsership', 'transmission', 'manufacturing_year', 'mileage(kmpl)', 'engine(cc)', 
            'max_power(bhp)', 'torque(Nm)']

# Numerical features & One-hot encoding for categorical
X = df[['seats', 'kms_driven', 'manufacturing_year', 'mileage(kmpl)', 'engine(cc)', 'max_power(bhp)', 'torque(Nm)']]
X = pd.concat([X, pd.get_dummies(df[['insurance_validity', 'fuel_type', 'ownsership', 'transmission']], drop_first=True)], axis=1)
y = df['price(in lakhs)']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"\nModel Performance:")
print(f"R2 Score: {r2:.4f}")
print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")

car_name                  str
registration_year         str
insurance_validity        str
fuel_type                 str
seats                   int64
kms_driven              int64
ownsership                str
transmission              str
manufacturing_year        str
mileage(kmpl)         float64
engine(cc)            float64
max_power(bhp)        float64
torque(Nm)            float64
price(in lakhs)       float64
dtype: object
                                            car_name registration_year  \
0                    2017 Mercedes-Benz S-Class S400            Jul-17   
1  2020 Nissan Magnite Turbo CVT XV Premium Opt BSVI            Jan-21   
2                       2018 BMW X1 sDrive 20d xLine            Sep-18   
3                           2019 Kia Seltos GTX Plus            Dec-19   
4                    2019 Skoda Superb LK 1.8 TSI AT            Aug-19   

  insurance_validity fuel_type  seats  kms_driven   ownsership transmission  \
0      Comprehensive    Petrol      5     

In [3]:
print(df.describe())

             seats    kms_driven  manufacturing_year  mileage(kmpl)  \
count  1499.000000    1499.00000         1499.000000    1499.000000   
mean      5.206137   53219.61441         2017.380921     187.284703   
std       0.636718   40404.02744            2.996381     516.597348   
min       4.000000     620.00000         2007.000000       7.810000   
25%       5.000000   30000.00000         2015.000000      16.100000   
50%       5.000000   49441.00000         2018.000000      18.760000   
75%       5.000000   70000.00000         2019.000000      21.400000   
max       8.000000  810000.00000         2023.000000    3996.000000   

         engine(cc)  max_power(bhp)    torque(Nm)  price(in lakhs)  
count  1.499000e+03    1.499000e+03  1.499000e+03      1499.000000  
mean   1.521934e+10    1.521934e+10  1.274292e+04       171.506071  
std    2.222352e+11    2.222352e+11  8.269170e+04      3540.885094  
min    6.700000e+01    6.700000e+01  1.900000e+01         1.000000  
25%    1.197000

In [4]:
# Let's inspect rows with price > 200 or engine(cc) > 10000 or mileage > 100
print("Outliers count:")
print("price > 100:", (df['price(in lakhs)'] > 100).sum())
print("mileage > 50:", (df['mileage(kmpl)'] > 50).sum())
print("engine > 6000:", (df['engine(cc)'] > 6000).sum())
print("torque > 1000:", (df['torque(Nm)'] > 1000).sum())

# Filter reasonable ranges for cars
df_clean = df[
    (df['price(in lakhs)'] <= 150) & 
    (df['mileage(kmpl)'] <= 40) & 
    (df['engine(cc)'] <= 6000) & 
    (df['kms_driven'] <= 300000)
]

print("Cleaned shape:", df_clean.shape)

X = df_clean[['seats', 'kms_driven', 'manufacturing_year', 'mileage(kmpl)', 'engine(cc)']]
X = pd.concat([X, pd.get_dummies(df_clean[['insurance_validity', 'fuel_type', 'ownsership', 'transmission']], drop_first=True)], axis=1)
y = df_clean['price(in lakhs)']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"\nFiltered Model Performance:")
print(f"R2 Score: {r2:.4f}")
print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")

Outliers count:
price > 100: 3
mileage > 50: 174
engine > 6000: 89
torque > 1000: 846
Cleaned shape: (1322, 14)

Filtered Model Performance:
R2 Score: 0.5317
MAE: 8.1240
RMSE: 12.6986


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Load dataset
df = pd.read_csv("Used Car Dataset.csv")

# Exploratory overview
print("Initial Shape:", df.shape)
print("Missing Values:\n", df.isnull().sum())

# Data cleaning
df_clean = df.copy()
if 'Unnamed: 0' in df_clean.columns:
    df_clean = df_clean.drop(columns=['Unnamed: 0'])

# Drop nulls
df_clean = df_clean.dropna()

# Convert manufacturing_year to numeric
df_clean['manufacturing_year'] = pd.to_numeric(df_clean['manufacturing_year'], errors='coerce')

# Filter extreme anomalies/outliers caused by corrupted data
df_clean = df_clean[
    (df_clean['price(in lakhs)'] > 0) & (df_clean['price(in lakhs)'] <= 150) &
    (df_clean['mileage(kmpl)'] > 0) & (df_clean['mileage(kmpl)'] <= 40) &
    (df_clean['engine(cc)'] > 500) & (df_clean['engine(cc)'] <= 6000) &
    (df_clean['kms_driven'] <= 300000)
]

print("Cleaned Shape:", df_clean.shape)

# Target & Features
X = df_clean[['manufacturing_year', 'kms_driven', 'mileage(kmpl)', 'engine(cc)', 'seats', 
              'fuel_type', 'transmission', 'ownsership']]
y = df_clean['price(in lakhs)']

# One-hot encoding
X = pd.get_dummies(X, columns=['fuel_type', 'transmission', 'ownsership'], drop_first=True)

# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Linear Regression Model
model = LinearRegression()
model.fit(X_train, y_train)

# Evaluation
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R² Score: {r2:.4f}")
print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")

Initial Shape: (1553, 15)
Missing Values:
 Unnamed: 0            0
car_name              0
registration_year     0
insurance_validity    0
fuel_type             0
seats                 0
kms_driven            0
ownsership            0
transmission          0
manufacturing_year    0
mileage(kmpl)         3
engine(cc)            3
max_power(bhp)        3
torque(Nm)            4
price(in lakhs)       0
dtype: int64
Cleaned Shape: (1322, 14)
R² Score: 0.5363
MAE: 8.0508
RMSE: 12.6368


In [6]:
df_clean['brand'] = df_clean['car_name'].str.split().str[1]
print("Top 15 brands:")
print(df_clean['brand'].value_counts().head(15))

# One hot encode top brands
top_brands = df_clean['brand'].value_counts().head(10).index
df_clean['brand_grouped'] = df_clean['brand'].apply(lambda x: x if x in top_brands else 'Other')

X_brand = df_clean[['manufacturing_year', 'kms_driven', 'mileage(kmpl)', 'engine(cc)', 'seats', 
                    'fuel_type', 'transmission', 'ownsership', 'brand_grouped']]
X_brand = pd.get_dummies(X_brand, columns=['fuel_type', 'transmission', 'ownsership', 'brand_grouped'], drop_first=True)
y_brand = df_clean['price(in lakhs)']

X_train, X_test, y_train, y_test = train_test_split(X_brand, y_brand, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R² Score with Brands: {r2:.4f}")
print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")

Top 15 brands:
brand
Maruti           274
Hyundai          251
Honda            174
Mercedes-Benz    122
BMW               76
Toyota            62
Audi              50
Tata              46
Mahindra          34
Ford              32
Renault           26
Kia               25
Volkswagen        24
Land              22
Nissan            19
Name: count, dtype: int64
R² Score with Brands: 0.5951
MAE: 7.0114
RMSE: 11.8089
